# Beacon qualification SLM — raw data → QLoRA adapter → evaluation table

Runs end to end on one free **Colab T4** session. Base model: `Qwen/Qwen2.5-1.5B-Instruct` (Apache-2.0).
1. Upload `beacon.zip` (create it locally with `git archive -o beacon.zip HEAD`).
2. Runtime → Change runtime type → **T4 GPU**. Run all cells.

Outputs: `slm/runs/colab/adapter/`, `slm/results/B.json`, `slm/results/C.json`, `slm/runs/colab/run_metadata.json` (peak VRAM, wall-clock, versions, data hashes).
System A (hosted model) is run locally with `python slm/evaluate.py run --system A --backend openai`, then `table` merges all three.

In [ ]:
import time, os, sys, json, subprocess
NOTEBOOK_STARTED = time.time()
from google.colab import files
if not os.path.exists('beacon'):
    up = files.upload()                      # choose beacon.zip
    subprocess.run(['unzip', '-q', next(iter(up)), '-d', 'beacon'], check=True)
os.chdir('beacon'); sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers', 'peft', 'bitsandbytes', 'accelerate', 'pydantic'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=False)  # Colab's preinstalled torchao is too old for peft
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)

## 1. Raw data → dataset (teacher scripts reproduce the raw batches; build.py validates, scores, splits)

In [ ]:
import glob
for script in sorted(glob.glob('slm/teacher/gen*.py'), key=lambda p: int(''.join(filter(str.isdigit, os.path.basename(p))))):
    subprocess.run([sys.executable, os.path.basename(script)], cwd='slm/teacher', check=True)
print(subprocess.run([sys.executable, 'slm/build.py'], capture_output=True, text=True).stdout[:1500])
stats = json.load(open('slm/data/stats.json')); print(stats['split_sizes'], stats['sha256'])

## 2. QLoRA fine-tune (NF4, rank 16, alpha 32, batch 1 × grad-accum 16; loss on the JSON answer only)

In [ ]:
import torch
subprocess.run([sys.executable, 'slm/train.py', '--out', 'slm/runs/colab', '--epochs', '2'], check=True)
meta = json.load(open('slm/runs/colab/run_metadata.json'))
print({k: meta[k] for k in ['train_seconds', 'total_seconds', 'peak_vram_gib', 'gpu']})
subprocess.run(['zip', '-qr', 'adapter_backup.zip', 'slm/runs/colab/adapter', 'slm/runs/colab/run_metadata.json'], check=True)
files.download('adapter_backup.zip')  # saved immediately so a disconnect during evaluation does not lose the training run

## 3. Evaluate B (few-shot base) and C (fine-tuned) on the frozen hand-labelled set

In [ ]:
EVAL = 'slm/eval/eval_set.jsonl' if os.path.exists('slm/eval/eval_set.jsonl') else 'slm/data/dev.jsonl'
print('evaluating on', EVAL, '(dev.jsonl = teacher labels: development numbers only)' if 'dev' in EVAL else '')
for system, extra in [('B', []), ('C', ['--adapter', 'slm/runs/colab/adapter'])]:
    subprocess.run([sys.executable, 'slm/evaluate.py', 'run', '--system', system, '--backend', 'hf', '--eval', EVAL, '--gpu-hourly', '0'] + extra, check=True)
print(subprocess.run([sys.executable, 'slm/evaluate.py', 'table'], capture_output=True, text=True).stdout)

## 4. Download the adapter, results and run metadata

In [ ]:
meta['notebook_seconds'] = round(time.time() - NOTEBOOK_STARTED); meta['eval_file'] = EVAL
json.dump(meta, open('slm/runs/colab/run_metadata.json', 'w'), indent=1)
subprocess.run(['zip', '-qr', 'beacon_run.zip', 'slm/runs/colab/adapter', 'slm/runs/colab/run_metadata.json', 'slm/results'], check=True)
files.download('beacon_run.zip')